In [9]:
%%time
#from sarma.ingestion.loader import load_pdf
from sarma.ingestion.knowledge_base import load_knowledge_base
from sarma.vectorstore.vectorstore import create_vector_store
from sarma.vectorstore.vectorstore import load_vector_store
from sarma.ingestion.splitter import split_documents
from sarma.retriever import create_retriever
from sarma.rag.rag import create_rag_chain
from sarma.prompts import rag_prompt
from sarma.llm import llm
import sarma.assistant as sas
from sarma.graph.workflow import create_sarma_graph

CPU times: total: 0 ns
Wall time: 0 ns


In [ ]:
%%time
try:
    db = load_vector_store()
    print("Loaded existing vector database")
    
except FileNotFoundError:
    print("Creating vector database...")
    #documents = load_pdf("../data/raw/RB209 Arable crops.pdf")
    documents = load_knowledge_base("../data/knowledge_base")
    chunks = split_documents(documents)
    db = create_vector_store(chunks)
    
retriever = create_retriever(db)
chain = create_rag_chain(retriever)

In [3]:
%%time
documents = load_knowledge_base("../data/knowledge_base")
chunks = split_documents(documents)
db = create_vector_store(chunks)

retriever = create_retriever(db)
chain = create_rag_chain(retriever)

found 2 PDF(s)

Loaing RB209 Arable crops.pdf
Loaing Sentinel-2-product-specifications-document-V14-9.pdf

Loaded 604 pages.


In [4]:
sarma_graph = create_sarma_graph(retriever, rag_prompt, llm)

In [8]:
result = sarma_graph.invoke({"question": "What are the recommended nitrogen rates for spring wheat?"})      

In [6]:
print(result["answer"])
print("\nSources:")
for c in result["citations"]:
    print("-", c)

The recommended nitrogen rates for spring-sown wheat, based on soil type and SNS Index, are as follows:  

- **Light sand soils**:  
  - SNS Index 0: 160 kg N/ha  
  - SNS Index 1: 130 kg N/ha  
  - SNS Index 2: 100 kg N/ha  
  - SNS Index 3: 70 kg N/ha  
  - SNS Index 4: 40 kg N/ha  
  - SNS Index 5: 0–40 kg N/ha  
  - SNS Index 6: 0 kg N/ha  

- **All other mineral soils**:  
  - SNS Index 0: 210a kg N/ha (exceeds N max limit in NVZs)  
  - SNS Index 1: 180 kg N/ha  
  - SNS Index 2: 150 kg N/ha  
  - SNS Index 3: 120 kg N/ha  
  - SNS Index 4: 70 kg N/ha  
  - SNS Index 5: 40 kg N/ha  
  - SNS Index 6: 0–40 kg N/ha  

- **Organic soils**:  
  - SNS Index 0: 120 kg N/ha  
  - SNS Index 1: 70 kg N/ha  
  - SNS Index 2: 40 kg N/ha  
  - SNS Index 3: 0–40 kg N/ha  

- **Peaty soils**:  
  - 0–40 kg N/ha  

**Note**: The N recommendation for "All other mineral soils" (210a kg N/ha) exceeds the N max limit in Nitrate Vulnerable Zones (NVZs). The N max limit is calculated for the entire fa

In [5]:
assistant = sas.SarmaAssistant(
    retriever,
    rag_prompt,
    llm
)

result = assistant.ask(
    "What are the recommended nitrogen rates for spring wheat?"
)


print("Answer:")
print(result["answer"])


print("\nSources:")

for index, source in enumerate(result["citations"]):
    print(
        f"{index+1}.{source}"
    )

Retrieval: 0.03 s
LLM: 57.15 s
Answer:
The recommended nitrogen rates for spring-sown wheat depend on soil type and SNS Index (a soil fertility indicator). Here are the rates from Table 4.18:  

- **Light sand soils**: 160, 130, 100, 70, 40, 0–40, 0 kg N/ha (for SNS Index 0–6).  
- **All other mineral soils**: 210a, 180, 150, 120, 70, 40, 0–40 kg N/ha (for SNS Index 0–6).  
- **Organic soils**: 120, 70, 40, 0–40 kg N/ha (for SNS Index 0–3).  
- **Peaty soils**: 0–40 kg N/ha.  

**Note**: The "a" in 210a indicates the recommendation exceeds the nitrogen maximum limit in Nitrate Vulnerable Zones (NVZs). For precise application timing and field-specific adjustments, refer to the full context.

Sources:
1.RB209 Arable crops.pdf, page 32


In [6]:
result = assistant.ask(
    "What is Sentinel-2?"
)


print("Answer:")
print(result["answer"])


print("\nSources:")

for index, source in enumerate(result["citations"]):
    print(
        f"{index+1}.{source}"
    )

Retrieval: 0.02 s
LLM: 23.01 s
Answer:
Sentinel-2 is an European Earth polar-orbiting satellite constellation consisting of two identical satellites (Sentinel-2A and 2B) that operate on a single orbit plane with a 180º phase difference. Each satellite hosts a Multi-Spectral Instrument (MSI) capable of capturing high-resolution imagery across the visible to shortwave infrared spectral range. The mission provides continuous, operational high-resolution data for global and sustained monitoring of Earth's land and coastal areas, supporting the GMES system with a high revisit frequency.

Sources:
1.Sentinel-2-product-specifications-document-V14-9.pdf, page 39
2.Sentinel-2-product-specifications-document-V14-9.pdf, page 26
3.Sentinel-2-product-specifications-document-V14-9.pdf, page 261
4.Sentinel-2-product-specifications-document-V14-9.pdf, page 522
5.Sentinel-2-product-specifications-document-V14-9.pdf, page 23
